# Vishwamitra · 1B Distillation Training

Train a `Llama-3.2-1B-Instruct` adapter to mimic the **Vishwamitra swarm-of-swarms** via supervised fine-tuning (SFT).

Runs on free Colab T4 in ~30–60 min.

## What this notebook does
1. Installs Unsloth + HF TRL
2. Uploads your `train.jsonl` and `val.jsonl` (generated by `generate_dataset.py`)
3. Loads Llama-3.2-1B-Instruct in 4-bit and attaches a LoRA adapter
4. Trains for 3 epochs with `SFTTrainer`
5. Plots training and validation loss
6. Saves the adapter and downloads it
7. Runs an inference sanity check comparing the model's output to the swarm's recommendation

## Before you run
- **Runtime → Change runtime type → T4 GPU** (free tier; A100 / L4 / V100 also work)
- Have `train.jsonl` and `val.jsonl` ready to upload, generated locally with:
  ```
  python generate_dataset.py --n 300
  ```


## Step 0. Verify GPU

In [ ]:
!nvidia-smi

## Step 1. Install Unsloth + HF TRL
Takes ~3 minutes.

In [ ]:
%%capture
!pip install -q "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install -q --no-deps "trl<0.13.0" peft accelerate bitsandbytes
!pip install -q datasets matplotlib pandas

## Step 2. Upload `train.jsonl` and `val.jsonl`

Upload BOTH files from your local `data/` folder when the picker opens.

In [ ]:
from google.colab import files
print('Upload BOTH train.jsonl and val.jsonl from your local data/ folder...')
uploaded = files.upload()

Verify the uploads:

In [ ]:
import os, json
for fname in ['train.jsonl', 'val.jsonl']:
    if not os.path.exists(fname):
        raise FileNotFoundError(f'{fname} not found — upload it via the cell above')
    with open(fname) as f:
        rows = [json.loads(line) for line in f if line.strip()]
    print(f'{fname}: {len(rows)} rows')
    sample = rows[0]
    print(f'  Keys: {list(sample.keys())}')
    print(f'  Messages: {[m["role"] for m in sample.get("messages", [])]}')

## Step 3. Training configuration

In [ ]:
MAX_SEQ_LEN      = 2048
LEARNING_RATE    = 2e-4
NUM_EPOCHS       = 3
PER_DEVICE_BATCH = 2
GRAD_ACCUM       = 4
WARMUP_STEPS     = 5
LOG_STEPS        = 5
EVAL_STEPS       = 25
SAVE_STEPS       = 50
SEED             = 42

OUTPUT_DIR = 'vishwamitra-1b-lora'
print(f'Effective batch size: {PER_DEVICE_BATCH * GRAD_ACCUM}')

## Step 4. Load base model in 4-bit (Unsloth)

In [ ]:
from unsloth import FastLanguageModel
import torch

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name      = 'unsloth/Llama-3.2-1B-Instruct',
    max_seq_length  = MAX_SEQ_LEN,
    dtype           = None,        # auto: bf16 on Ampere+, fp16 on T4
    load_in_4bit    = True,
)
print('Base model:', model.config.name_or_path)
print('Vocab size:', tokenizer.vocab_size)

## Step 5. Attach LoRA adapter

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r              = 16,
    lora_alpha     = 16,
    target_modules = ['q_proj', 'k_proj', 'v_proj', 'o_proj',
                      'gate_proj', 'up_proj', 'down_proj'],
    lora_dropout   = 0.0,
    bias           = 'none',
    use_gradient_checkpointing = 'unsloth',
    random_state   = SEED,
)
model.print_trainable_parameters()

## Step 6. Apply chat template + format dataset

In [ ]:
from datasets import load_dataset
from unsloth.chat_templates import get_chat_template

# Llama-3.2 uses the same chat template as Llama-3.1.
tokenizer = get_chat_template(tokenizer, chat_template='llama-3.1')

ds = load_dataset(
    'json',
    data_files = {'train': 'train.jsonl', 'validation': 'val.jsonl'},
)
print(ds)

def format_chat(examples):
    convos = examples['messages']
    texts = [
        tokenizer.apply_chat_template(c, tokenize=False, add_generation_prompt=False)
        for c in convos
    ]
    return {'text': texts}

ds = ds.map(format_chat, batched=True)

print('\nFirst formatted example (truncated):\n')
print(ds['train'][0]['text'][:1500])

## Step 7. Configure SFTTrainer

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments

trainer = SFTTrainer(
    model              = model,
    tokenizer          = tokenizer,
    train_dataset      = ds['train'],
    eval_dataset       = ds['validation'],
    dataset_text_field = 'text',
    max_seq_length     = MAX_SEQ_LEN,
    dataset_num_proc   = 2,
    packing            = False,
    args = TrainingArguments(
        output_dir                  = OUTPUT_DIR,
        num_train_epochs            = NUM_EPOCHS,
        per_device_train_batch_size = PER_DEVICE_BATCH,
        gradient_accumulation_steps = GRAD_ACCUM,
        warmup_steps                = WARMUP_STEPS,
        learning_rate               = LEARNING_RATE,
        fp16                        = not torch.cuda.is_bf16_supported(),
        bf16                        = torch.cuda.is_bf16_supported(),
        logging_steps               = LOG_STEPS,
        eval_strategy               = 'steps',
        eval_steps                  = EVAL_STEPS,
        save_strategy               = 'steps',
        save_steps                  = SAVE_STEPS,
        save_total_limit            = 2,
        optim                       = 'adamw_8bit',
        weight_decay                = 0.01,
        lr_scheduler_type           = 'cosine',
        seed                        = SEED,
        report_to                   = 'none',
    ),
)

## Step 8. Train

Watch for: training loss should drop from ~3.0 → ~0.4 over 200–400 steps. If it stays flat, something's wrong with the dataset format.

In [ ]:
import time
t0 = time.time()
stats = trainer.train()
print(f'\nTotal training time: {(time.time() - t0) / 60:.1f} min')

## Step 9. Plot training + validation loss

This is the curve that goes into the README.

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

log = trainer.state.log_history
df = pd.DataFrame(log)

train_rows = df.dropna(subset=['loss'])             if 'loss' in df.columns      else pd.DataFrame()
eval_rows  = df.dropna(subset=['eval_loss'])        if 'eval_loss' in df.columns else pd.DataFrame()

fig, ax = plt.subplots(figsize=(10, 5))
if not train_rows.empty:
    ax.plot(train_rows['step'], train_rows['loss'],
            label='train', color='#3b82f6', linewidth=2)
if not eval_rows.empty:
    ax.plot(eval_rows['step'], eval_rows['eval_loss'],
            label='validation', color='#fbbf24', marker='o', linewidth=2)
ax.set_xlabel('Training step')
ax.set_ylabel('Cross-entropy loss')
ax.set_title('Vishwamitra distillation — SFT loss')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('loss_curve.png', dpi=150, bbox_inches='tight')
plt.show()

df.to_csv('training_log.csv', index=False)
print('Saved: loss_curve.png, training_log.csv')

## Step 10. Save the LoRA adapter + download all artefacts

In [ ]:
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f'Saved adapter to {OUTPUT_DIR}/')

# Zip and trigger downloads
!cd {OUTPUT_DIR} && zip -qr ../{OUTPUT_DIR}.zip .

from google.colab import files
files.download(f'{OUTPUT_DIR}.zip')
files.download('loss_curve.png')
files.download('training_log.csv')

## Step 11. Inference sanity check

Loads one held-out validation example, generates a recommendation with the trained model, and compares it to the swarm's actual recommendation. The output of this cell is the "did distillation actually work" smoke test.

In [ ]:
import json, re

FastLanguageModel.for_inference(model)

with open('val.jsonl') as f:
    val_rows = [json.loads(line) for line in f if line.strip()]

example = val_rows[0]
input_messages = example['messages'][:-1]            # everything except the assistant's response
target         = example['messages'][-1]['content']  # the swarm's recommendation we want to mimic

prompt = tokenizer.apply_chat_template(input_messages, tokenize=False, add_generation_prompt=True)
inputs = tokenizer(prompt, return_tensors='pt').to('cuda')

outputs = model.generate(
    **inputs,
    max_new_tokens     = 512,
    temperature        = 0.4,
    do_sample          = True,
    repetition_penalty = 1.1,
)
generated = tokenizer.decode(
    outputs[0][inputs.input_ids.shape[1]:],
    skip_special_tokens=True,
)

print('=' * 60)
print('MODEL OUTPUT:')
print('=' * 60)
print(generated[:1500])
print()
print('=' * 60)
print("TARGET (swarm's actual recommendation):")
print('=' * 60)
print(target[:1500])

# Numerical diff if the model produced parseable JSON
try:
    m = re.search(r'\{.*\}', generated, re.DOTALL)
    gen_json = json.loads(m.group(0)) if m else None
    tgt_json = json.loads(target)
    if gen_json and 'action_vector' in gen_json:
        print('\n=== ACTION VECTOR DIFF ===')
        gv = gen_json['action_vector']
        tv = tgt_json['action_vector']
        diffs = []
        for k in tv:
            g = float(gv.get(k, 0))
            t = float(tv[k])
            mark = '✓' if abs(g - t) < 0.15 else '≈' if abs(g - t) < 0.3 else '✗'
            print(f'  {k:22s}  target={t:.2f}  model={g:.2f}  diff={abs(g-t):.2f}  {mark}')
            diffs.append(abs(g - t))
        print(f'\nMean Absolute Error across interventions: {sum(diffs)/len(diffs):.3f}')
        print('  (target: <0.15 = excellent · <0.30 = OK · >=0.30 = under-trained)')
except Exception as e:
    print(f'\nCould not parse model output as JSON: {e}')

## Step 12. (Optional) Push the adapter to Hugging Face Hub

Uncomment and run this if you want the adapter discoverable at `<your-username>/vishwamitra-1b-lora`. Get a write token from <https://huggingface.co/settings/tokens>.

In [ ]:
# from huggingface_hub import login
# login(token='hf_xxx')                                      # paste your HF write token
#
# REPO = 'your-username/vishwamitra-1b-lora'                 # change me
# model.push_to_hub(REPO, save_method='lora')
# tokenizer.push_to_hub(REPO)

## ✅ Done

You now have:

| Artefact | What it's for |
|---|---|
| `vishwamitra-1b-lora.zip` | The trained LoRA adapter |
| `loss_curve.png` | Embed in `README_HACKATHON.md` § 5 (Results) |
| `training_log.csv` | Full per-step training metrics |

### Next steps (back on your laptop)

1. Drop `loss_curve.png` into `docs/img/` of the main repo
2. Run `python evaluation/eval_distilled.py` (next deliverable) to compare the trained model vs random/zero baselines on `DropoutCommonsEnv` — produces `reward_curve.png`
3. Replace the `>>> TODO` placeholders in `README_HACKATHON.md` § 5 with the numbers from this notebook + the eval script
4. Record the 2-min video walkthrough
5. Push the adapter to HF Hub (Step 12 above) and link it in the README
